<a href="https://colab.research.google.com/github/Agasthiya-09/UG/blob/main/Honors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# INSTALL
# =========================
!pip install gradio tensorflow scikit-learn pandas numpy plotly

# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np
import gradio as gr
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px
import plotly.graph_objects as go

# =========================
# GLOBAL OBJECTS
# =========================
model = None
scaler = None
kmeans = None
feature_names = [
    "recency", "frequency", "monetary",
    "avg_interval", "season_encoded"
]

# =========================
# FEATURE ENGINEERING
# =========================
def get_season(month):
    if month in [3,4,5]:
        return 0  # Summer
    elif month in [6,7,8,9]:
        return 1  # Monsoon
    elif month in [10,11]:
        return 2  # Post-Monsoon
    else:
        return 3  # Winter

def preprocess(df):

    df.columns = df.columns.str.strip().str.replace(" ", "_")

    df["Purchase_Date"] = pd.to_datetime(df["Purchase_Date"])

    # ---------------------
    # RFM FEATURES
    # ---------------------
    df["monetary"] = df["Purchase_Amount_(USD)"]
    df["frequency"] = df["Frequency_of_Purchases"].astype("category").cat.codes + 1

    last_date = df["Purchase_Date"].max()
    df["recency"] = (last_date - df["Purchase_Date"]).dt.days

    # ---------------------
    # PURCHASE INTERVAL
    # ---------------------
    df = df.sort_values(by=["Customer_ID", "Purchase_Date"])

    df["prev_date"] = df.groupby("Customer_ID")["Purchase_Date"].shift(1)
    df["interval"] = (df["Purchase_Date"] - df["prev_date"]).dt.days
    df["interval"] = df["interval"].fillna(df["interval"].mean())

    df["avg_interval"] = df.groupby("Customer_ID")["interval"].transform("mean")

    # ---------------------
    # SEASON
    # ---------------------
    df["season"] = df["Purchase_Date"].dt.month.apply(get_season)
    df["season_encoded"] = df["season"]

    # ---------------------
    # LABEL (Trending likelihood)
    # ---------------------
    df["target"] = (
        (df["monetary"] > df["monetary"].median()) &
        (df["frequency"] > df["frequency"].median())
    ).astype(int)

    return df

# =========================
# MODEL TRAINING (MLP)
# =========================
def train_model(file):

    global model, scaler, kmeans

    df = pd.read_csv(file.name)
    df = preprocess(df)

    X = df[feature_names]
    y = df["target"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ---------------------
    # DEEP MODEL (MLP)
    # ---------------------
    model = keras.Sequential([
        keras.layers.Dense(128, activation="relu", input_shape=(len(feature_names),)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_scaled, y, epochs=20, verbose=0)

    # ---------------------
    # SEGMENTATION
    # ---------------------
    kmeans = KMeans(n_clusters=4, random_state=42)
    df["segment"] = kmeans.fit_predict(X_scaled)

    fig = px.histogram(df, x="segment", title="Customer Segmentation")

    return fig

# =========================
# RL DECISION (Bandit Logic)
# =========================
def rl_decision(prob, season):

    # Contextual decision (season-aware)
    if prob > 0.8:
        return "High Stock + Premium Ads"
    elif prob > 0.5:
        return "Discount Campaign"
    else:
        return "Retention Campaign"

# =========================
# RULE + BAYES VALIDATION
# =========================
def rule_validation(recency, frequency):

    if recency > 200:
        return "At Risk"
    elif frequency > 20:
        return "Loyal"
    else:
        return "Regular"

# =========================
# EXPLAINABLE AI
# =========================
def integrated_gradients(inputs, steps=50):

    baseline = np.zeros_like(inputs)

    scaled_inputs = [
        baseline + (i/steps)*(inputs-baseline)
        for i in range(steps+1)
    ]

    grads = []
    for inp in scaled_inputs:
        inp = tf.convert_to_tensor(inp.reshape(1, -1), dtype=tf.float32)

        with tf.GradientTape() as tape:
            tape.watch(inp)
            pred = model(inp)

        grad = tape.gradient(pred, inp)
        grads.append(grad.numpy())

    avg_grads = np.mean(grads, axis=0)
    return (inputs - baseline) * avg_grads[0]

# =========================
# COUNTERFACTUAL
# =========================
def counterfactual(input_data):

    cf = input_data.copy()

    # Increase monetary artificially
    cf[2] *= 1.5

    return cf

# =========================
# PREDICTION PIPELINE
# =========================
def predict(rec, freq, mon, interval, season):

    input_data = np.array([[rec, freq, mon, interval, season]])
    scaled = scaler.transform(input_data)

    prob = float(model.predict(scaled)[0][0])

    decision = rl_decision(prob, season)
    rule = rule_validation(rec, freq)

    # ---------------------
    # EXPLANATION
    # ---------------------
    ig = integrated_gradients(scaled[0])

    importance = pd.DataFrame({
        "Feature": feature_names,
        "Importance": ig
    })

    fig_imp = px.bar(importance, x="Feature", y="Importance",
                     title="Feature Importance")

    # ---------------------
    # COUNTERFACTUAL
    # ---------------------
    cf_input = counterfactual(input_data[0])
    cf_scaled = scaler.transform([cf_input])
    cf_prob = float(model.predict(cf_scaled)[0][0])

    # ---------------------
    # GAUGE
    # ---------------------
    gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=prob,
        title={'text': "Trend Probability"},
        gauge={'axis': {'range': [0,1]}}
    ))

    return gauge, decision, rule, fig_imp, cf_prob

# =========================
# TREND + SEASON ANALYSIS
# =========================
def analyze_trends(file):

    df = pd.read_csv(file.name)
    df.columns = df.columns.str.strip().str.replace(" ", "_")

    df["Purchase_Date"] = pd.to_datetime(df["Purchase_Date"])

    trend_df = df.groupby([
        pd.Grouper(key="Purchase_Date", freq="M"),
        "Product_Name"
    ])["Quantity"].sum().reset_index()

    latest = trend_df["Purchase_Date"].max()

    trending = trend_df[
        trend_df["Purchase_Date"] == latest
    ].sort_values(by="Quantity", ascending=False)

    fig1 = px.bar(trending, x="Product_Name", y="Quantity",
                  title="Trending Products")

    df["season"] = df["Purchase_Date"].dt.month.apply(get_season)

    season_df = df.groupby(
        ["season", "Product_Name"]
    )["Quantity"].sum().reset_index()

    fig2 = px.bar(season_df, x="season", y="Quantity",
                  color="Product_Name",
                  title="Seasonal Demand")

    return fig1, fig2

# =========================
# GRADIO UI
# =========================
with gr.Blocks() as demo:

    gr.Markdown("# 🚀 AI Product Trend Prediction System")

    with gr.Tab("Train Model"):
        file_input = gr.File()
        btn = gr.Button("Train")
        out = gr.Plot()
        btn.click(train_model, file_input, out)

    with gr.Tab("Predict Trend"):

        rec = gr.Number(label="Recency")
        freq = gr.Number(label="Frequency")
        mon = gr.Number(label="Monetary")
        interval = gr.Number(label="Avg Purchase Interval")
        season = gr.Number(label="Season (0-3)")

        btn2 = gr.Button("Predict")

        g = gr.Plot()
        d = gr.Textbox(label="RL Decision")
        r = gr.Textbox(label="Rule Segment")
        imp = gr.Plot()
        cf = gr.Textbox(label="Counterfactual Probability")

        btn2.click(
            predict,
            inputs=[rec, freq, mon, interval, season],
            outputs=[g, d, r, imp, cf]
        )

    with gr.Tab("Trend Analysis"):
        f2 = gr.File()
        b2 = gr.Button("Analyze")

        t1 = gr.Plot()
        t2 = gr.Plot()

        b2.click(analyze_trends, f2, [t1, t2])

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d8a0a66ab401ba344c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1. Recency (rec)
Days since last purchase
📉 Lower = more recent customer

👉 Typical values:

0–30 → very active
30–100 → moderate
100–300 → inactive
2. Frequency (freq)
Number of purchases

👉 Typical values:

1–5 → low
5–20 → medium
20+ → high (loyal)
3. Monetary (mon)
Total spending (USD or your dataset unit)

👉 Typical values:

10–100 → low
100–500 → medium
500+ → high-value
4. Avg Purchase Interval (interval)
Average days between purchases

👉 Typical values:

1–10 → frequent buyer
10–30 → moderate
30+ → occasional
5. Season (season)

From your encoding:

0 = Summer
1 = Monsoon
2 = Post-Monsoon
3 = Winter